In [1]:
import numpy as np
import torch
from torch import nn
import torch.optim as optim
import matplotlib.pyplot as plt

%matplotlib inline
torch.set_printoptions(sci_mode=False,precision=4)
torch.manual_seed(123)
torch.random.manual_seed(123)
generator = torch.Generator().manual_seed(123)

In [2]:
context_length = 96
batch_size = 32
embd_dim = 32
key_value_embd_dim = 4
iteration_count = 100000

In [3]:
with open('../data/amir_khan.txt', 'r', encoding = 'utf-8') as file:
    records = file.read()

In [4]:
data = ''.join(records)
data = '_' * context_length + data
unique_text = sorted(list(set(data)))
vocab_size = len(unique_text)

In [5]:
stoi = {text: index for index, text in enumerate(unique_text)}
itos = {index: text for index, text in enumerate(unique_text)}

encoder = lambda char_array: [stoi[char] for char in ''.join(char_array)]
decoder = lambda num_array: [itos[num] for num in num_array]

In [6]:
x_numpy = []
y_numpy = []
temp_data = data

for t in range(len(temp_data) - context_length - 1):
    x_numpy.append(encoder(temp_data[t : t + context_length]))
    y_numpy.append(stoi[temp_data[t + context_length]])

x = torch.tensor(x_numpy, dtype=torch.long)
y = torch.tensor(y_numpy, dtype=torch.long)



N = x.shape[0]
train_ratio = 0.8
val_ratio = 0.1

train_end = int(train_ratio * N)
val_end = int((train_ratio + val_ratio) * N)

x_train, y_train = x[:train_end], y[:train_end]
x_val, y_val = x[train_end:], y[train_end:]

In [7]:
criterion = nn.CrossEntropyLoss()
embedding = nn.Embedding(vocab_size, embd_dim)
logit_layer = nn.Linear(context_length * embd_dim, vocab_size)

In [8]:
def evaluate(x_data, y_data):
    embedding.eval()
    logit_layer.eval()

    with torch.no_grad():
        embd = embedding(x_data)
        embd = embd.view(embd.size(0), -1)
        logits = logit_layer(embd)

        val_loss = criterion(logits, y_data)

        probs = torch.softmax(logits, dim=1)
        predictions = torch.argmax(probs, dim=1)

        accuracy = (predictions == y_data).float().mean()

    embedding.train()
    logit_layer.train()

    return val_loss.item(), accuracy.item()

In [9]:
embedding.eval()
logit_layer.eval()

optimizer = optim.Adam(
    list(embedding.parameters()) +
    list(logit_layer.parameters()),
    lr=0.005
)

for epoch in range(iteration_count):
    idx = torch.randint(0, len(x_train), (batch_size,), generator=generator)
    rand_x = x_train[idx]
    rand_y = y_train[idx]

    optimizer.zero_grad(set_to_none=True)

    embd = embedding(rand_x)
    embd = embd.view(embd.size(0), -1)
    logits = logit_layer(embd)
    loss = criterion(logits, rand_y)

    if epoch % 10000 == 0:
        print(loss.item(), evaluate(x_val, y_val)[0])

    loss.backward()
    optimizer.step()


4.159954071044922 4.335694313049316
2.6696531772613525 2.5685336589813232
2.6266536712646484 2.570849895477295
2.135374069213867 2.57407808303833
1.7737693786621094 2.5915122032165527
2.123340129852295 2.589186906814575
2.235797166824341 2.5787899494171143
2.3586552143096924 2.622755765914917
2.4993977546691895 2.5951170921325684
2.2848153114318848 2.5796730518341064
